In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
from urllib.parse import urlparse, parse_qs
import re
import time
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup
import datetime
import os
from selenium import webdriver
from time import sleep
import pandas as pd
from collections import deque
from urllib.parse import urldefrag
from urllib.parse import urlsplit, urlunsplit
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'LV CBOL' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)



Running LV CBOL Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
#Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder,
         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert
         }
chromeOptions.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


In [4]:
#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

        # regulatorName + ' 1': 'https://uzraudziba.bank.lv/en/market/alternative-investment-fund-managers/',
        # regulatorName + ' 2': 'https://uzraudziba.bank.lv/en/market/insurance-companies/',
        # regulatorName + ' 3': 'https://uzraudziba.bank.lv/en/market/insurance-intermediaries/',
        # regulatorName + ' 4': 'https://uzraudziba.bank.lv/en/market/financial-instruments-market/',
        # regulatorName + ' 5': 'https://uzraudziba.bank.lv/en/market/financial-holdings/',
        # regulatorName + ' 6': 'https://uzraudziba.bank.lv/en/market/investment-service-providers/',
        # #regulatorName + ' 7 1': 'https://uzraudziba.bank.lv/en/market/investment-management-companies/',
        #  regulatorName + ' 7': 'https://uzraudziba.bank.lv/en/market/',
        # #regulatorName + ' 7 2': 'https://uzraudziba.bank.lv/en/market/investment-management-companies/foreign-funds/',
        # regulatorName + ' 8': 'https://uzraudziba.bank.lv/en/market/',
        # regulatorName + ' 9': 'https://uzraudziba.bank.lv/en/market/',
        # regulatorName + ' 10': 'https://uzraudziba.bank.lv/en/market/credit-institutions/',
        regulatorName + ' 11': 'https://uzraudziba.bank.lv/en/market/payment-service-providers/',
        # regulatorName + ' 12': 'https://uzraudziba.bank.lv/en/market/pension-funds/',
        # regulatorName + ' 13': 'https://uzraudziba.bank.lv/en/market/',


        }



Typology={

    regulatorName + ' 1': 'Investment service providers',
    regulatorName + ' 2': 'Insurance companies',
    regulatorName + ' 3': 'Insurance Intermediaries',
    regulatorName + ' 4': 'Financial instruments market',
    regulatorName + ' 5': 'Financial holdings',
    regulatorName + ' 6': 'Investment service providers',
    regulatorName + ' 7': 'Investment management companies',
    # regulatorName + ' 7 1': 'Investment management companies',
    # regulatorName + ' 7 2': 'Investment management companies',
    regulatorName + ' 8': 'Crowdfunding service providers',
    regulatorName + ' 9': 'Co-operative Credit Unions',
    regulatorName + ' 10': 'Credit institutions',
    regulatorName + ' 11': 'Payment service providers',
    regulatorName + ' 12': 'Pension Funds',
    regulatorName + ' 13': 'Foreign exchange trading companies',

        }

sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')


skip_by_reg = {
    f"{regulatorName} 1": [
        "managers-from-eea",
    ],
    f"{regulatorName} 2":[
        "insolvent-insurance-companies",
        "insurance-companies-in-liquidation",
        "liquidated-reorganized-insurance-companies",
        "service-providers-from-the-eea"
    ],
    f"{regulatorName} 3": [
        "service-providers-from-the-eea"
    ],
    f"{regulatorName} 4":[
        "central-securities-depositories-from-the-eea",
        "securities-offerings-by-member-states"
    ],

    f"{regulatorName} 6":[
        "registered-foreign-investment-service-providers",
        "service-providers-from-the-eea",
        "tied-agents-of-eea-investment-service-providers"
    ],
    f"{regulatorName} 7":[
        "sanctions",
        "pension-funds",
        "credit-institutions",
        "crowdfunding-service-providers",
        "investment-service-providers",
        "financial-instruments-market",
        "insurance-companies",
        "crypto-asset-market",
        "alternative-investment-fund-managers",
        "insurance-intermediaries",
        "financial-holdings",
        # "investment-management-companies",
        "co-operative-credit-unions",
        "payment-service-providers",
        "foreign-exchange-trading-companies"

    ],
        f"{regulatorName} 8":[
        "sanctions",
        "pension-funds",
        "credit-institutions",
        # "crowdfunding-service-providers",
        "investment-service-providers",
        "financial-instruments-market",
        "insurance-companies",
        "crypto-asset-market",
        "alternative-investment-fund-managers",
        "insurance-intermediaries",
        "financial-holdings",
        "investment-management-companies",
        "co-operative-credit-unions",
        "payment-service-providers",
        "foreign-exchange-trading-companies"
    ],
        f"{regulatorName} 9":[
        "sanctions",
        "pension-funds",
        "credit-institutions",
        "crowdfunding-service-providers",
        "investment-service-providers",
        "financial-instruments-market",
        "insurance-companies",
        "crypto-asset-market",
        "alternative-investment-fund-managers",
        "insurance-intermediaries",
        "financial-holdings",
        "investment-management-companies",
        #"co-operative-credit-unions",
        "payment-service-providers",
        "foreign-exchange-trading-companies"
    ],
        f"{regulatorName} 10":[
        "credit-institutions-in-reorganization",
        "credit-institutions-in-liquidation",
        "service-providers-from-the-eea"
    ],
        f"{regulatorName} 11":[
        "service-providers-from-the-eea"
    ],
        f"{regulatorName} 13":[
        "sanctions",
        "pension-funds",
        "credit-institutions",
        "crowdfunding-service-providers",
        "investment-service-providers",
        "financial-instruments-market",
        "insurance-companies",
        "crypto-asset-market",
        "alternative-investment-fund-managers",
        "insurance-intermediaries",
        "financial-holdings",
        "investment-management-companies",
        "co-operative-credit-unions",
        "payment-service-providers",
        #"foreign-exchange-trading-companies"
    ],

}

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
})

In [5]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict
    
def get_soup(url, tries=3, sleep=0.5):
    last = None
    for i in range(tries):
        try:
            r = session.get(url, timeout=30,verify=False)
            r.raise_for_status()
            return BeautifulSoup(r.text, "lxml")
        except Exception as e:
            last = e
            time.sleep(sleep * (i + 1))
    raise last

def norm_label(s):
    return re.sub(r"\s+", " ", (s or "").strip()).rstrip(":").strip().lower()

def extract_post_urls(soup, base_url):
    post_urls = []
    for a in soup.select("div.posts-block a[href]"):
        href = a.get("href")
        if href:
            post_urls.append(urljoin(base_url, href))
    return list(dict.fromkeys(post_urls))


def extract_post_titles(soup, base_url=None):
    """
    Extract company titles from the posts list.
    Even if href points back to the same category (cat_url), we still take the
    visible text (e.g. <div class="title-content">...</div>) as the company title.
    Returns {resolved_url: title}.
    """
    title_url = {}

    for a in soup.select("div.posts-block a[href]"):
        href = (a.get("href") or "").strip()
        if not href:
            continue

        url = urljoin(base_url, href) if base_url else href

        title_el = a.select_one("div.title-content")
        title = (title_el.get_text(" ", strip=True) if title_el else a.get_text(" ", strip=True)).strip()

        if not title:
            title = url  # only if truly no visible title

        title_url[url] = title

    return title_url

def extract_l_pages(soup, base_url):
    l_vals = set()
    for a in soup.select("a[href*='?l=']"):
        href = a.get("href")
        if not href:
            continue
        abs_href = urljoin(base_url, href)
        qs = parse_qs(urlparse(abs_href).query)
        if "l" in qs:
            try:
                l_vals.add(int(qs["l"][0]))
            except Exception:
                pass
    return sorted(l_vals)



In [6]:
from collections import deque
from urllib.parse import urlsplit, urlunsplit, urldefrag

def norm_url(u: str) -> str:
    u = urldefrag(u)[0]
    p = urlsplit(u)
    return urlunsplit((p.scheme, p.netloc, p.path.rstrip("/") + "/", "", ""))  # drop query/fragment

# 1) categories
for reg in regdict:
    print(f'------ Working with {reg} -------')
    SKIP_SUBSTRINGS = skip_by_reg.get(reg, [])
    soup = get_soup(regdict[reg])
    cat_as = soup.select("div.categories-list a[href]")
    category_urls = []
    for a in cat_as:
        href = a.get("href")
        if href:
            category_urls.append(urljoin(regdict[reg], href))
    category_urls = list(dict.fromkeys(category_urls))
    if reg == regulatorName+' 7':
        category_urls.append('https://uzraudziba.bank.lv/en/market/investment-management-companies/foreign-funds/')

    visited_posts = set()

    # 2) for each category: dive subcategories + paginate (?l=1,2,...) and scrape posts
    for cat_url in category_urls:
        cat_queue = deque([norm_url(cat_url)])
        seen_cats = set()

        while cat_queue:
            cur_cat = norm_url(cat_queue.popleft())
            if cur_cat in seen_cats:
                continue
            if SKIP_SUBSTRINGS and any  (s in cur_cat for s in SKIP_SUBSTRINGS):
                continue
            seen_cats.add(cur_cat)

            first_soup = get_soup(cur_cat)

            # enqueue subcategories (but never enqueue ?l= pages)
            for a in first_soup.select("div.categories-list a[href]"):
                href = a.get("href")
                if not href:
                    continue
                sub_url = norm_url(urljoin(cur_cat, href))
                if sub_url not in seen_cats:
                    cat_queue.append(sub_url)

            base_cur = cur_cat  # already normalized with trailing "/"

            # build listing pages for this category
            l_pages = extract_l_pages(first_soup, cur_cat)
            if l_pages:
                list_pages = [base_cur + f"?l={i}" for i in range(1, max(l_pages) + 1)]
            else:
                list_pages = []
                prev_fp = None
                for i in range(1, 200):
                    u = base_cur + f"?l={i}"
                    s = get_soup(u)
                    urls = extract_post_urls(s, u)
                    fp = tuple(urls[:10])
                    if not urls or fp == prev_fp:
                        break
                    prev_fp = fp
                    list_pages.append(u)
                if not list_pages:
                    list_pages = [cur_cat]

            # scrape posts from listing pages
            for page_url in list_pages:
                soup = first_soup if page_url == cur_cat else get_soup(page_url)
                post_urls = extract_post_urls(soup, page_url)
                title_urls = extract_post_titles(soup)

                for post_url in post_urls:
                    if post_url in visited_posts:
                        continue
                    visited_posts.add(post_url)

                    dsoup = get_soup(post_url)
                    h2 = dsoup.select_one("h2")
                    title = (h2.get_text(" ", strip=True) if h2 else "").strip()
                    if not title.strip():
                        title = title_urls[post_url]

                    pairs = {}
                    for block in dsoup.select("div.row div.info-block"):
                        for lab in block.select(".market-item.label"):
                            label = norm_label(lab.get_text(" ", strip=True))
                            val_el = lab.find_next_sibling(
                                lambda t: t.name == "div"
                                and "market-item" in (t.get("class") or [])
                                and "label" not in (t.get("class") or [])
                            )
                            value = val_el.get_text(" ", strip=True) if val_el else ""
                            if label:
                                pairs[label] = value

                    row = {k: "" for k in sqldict.keys()}
                    print(title)
                    row["Name"] = title
                    row["Address_1"] = pairs.get("legal address", "") or pairs.get("address", "")
                    row["Website"] = pairs.get("website", "") or pairs.get("web site", "") or pairs.get("web", "")
                    row["Email"] = pairs.get("e-mail", "") or pairs.get("email", "")
                    row["Phone"] = pairs.get("phone", "") or pairs.get("telephone", "") or pairs.get("tel.", "")
                    row["Fax"] = pairs.get("fax", "")
                    row["Check"] = post_url
                    row["ListProcessDate"] = processdate
                    
                    row['Cntry']='LV'
                    row['RegCtry']= reg.split()[0]
                    row['RegCode']= reg.split()[1]
                    row['ListCode']= reg.split()[2]
                    row['ListName'] = Typology[reg]
                    row['Typology']
                    row['RegulationType'] = ('Regulated')

                    for k in sqldict.keys():
                        sqldict[k].append(row.get(k, ""))


------ Working with LV CBOL 11 -------
Latvijas tirdzniecības flotes jūrnieku arodbiedrību KKS "Jūrnieku forums"
Zosēnu Kooperatīvā krājaizdevu sabiedrība
"Swedbank" AS
Akciju sabiedrība "Citadele banka"
Akciju sabiedrība "Reģionālā investīciju banka"
Akciju sabiedrība "Rietumu Banka"
AS "Industra Bank"
AS Magnetiq Bank
AS "SEB banka"
BluOr Bank AS
Signet Bank AS
AP OPERATIONS SIA
Ltd. "Mobilly"
SIA "Paytegra"
SIA "Transact Pro"
SIA "xpate"
SIA GR8 PAY
SIA Mintos Payments
SIA Pace FS
TigSiPay SIA
Ltd Andele Mandele PAY
"MAXIMA Latvija" SIA
DKV EURO SERVICE GmbH + Co. KG
Edenred Corporate Payment
Ltd "NESTE LATVIJA"
Ltd. "Latvijas Mobilais Telefons"
RIMI LATVIA, SIA
Sabiedrība ar ierobežotu atbildību "Tele2"
SIA "BITE Latvija"
SIA "Lieliska dāvana"
SIA “Circle K Latvia”
Ltd. "SEMFOPAY"
SIA "JOOL PAY"
SIA "SOLLO LV"
VAS "Latvijas Pasts"
"Swedbank" AS
Akciju sabiedrība "Citadele banka"
AS Magnetiq Bank
AS "SEB banka"
BluOr Bank AS
Luminor Bank AS Latvijas filiāle


In [7]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
zip_re = re.compile(r"\bLV[-\s]?\d{4}\b", re.IGNORECASE)

# after you build df = pd.DataFrame(sqldict)
df["Zip"] = df["Address_1"].fillna("").str.extract(r"(\bLV[-\s]?\d{4}\b)", expand=False).fillna("")
df["Zip"] = df["Zip"].str.upper().str.replace("LV ", "LV-", regex=False)
# df["Name"] = df["Name"].fillna("").str.strip()
df = df[df["Name"] != ""].reset_index(drop=True)
df.to_excel(filename, index=False)
driver.quit()
sleep(3)

In [8]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,,,,Latvijas tirdzniecības flotes jūrnieku arodbie...,,,,,...,,,,,,,,,,https://uzraudziba.bank.lv/en//market/payment-...
1,,,,,,Zosēnu Kooperatīvā krājaizdevu sabiedrība,,,,,...,,,,,,,,,,https://uzraudziba.bank.lv/en//market/payment-...
2,,,,,,"""Swedbank"" AS",,,,,...,,,,,,,,,,https://uzraudziba.bank.lv/en//market/payment-...
3,,,,,,"Akciju sabiedrība ""Citadele banka""",,,,,...,,,,,,,,,,https://uzraudziba.bank.lv/en//market/payment-...
4,,,,,,"Akciju sabiedrība ""Reģionālā investīciju banka""",,,,,...,,,,,,,,,,https://uzraudziba.bank.lv/en//market/payment-...
5,,,,,,"Akciju sabiedrība ""Rietumu Banka""",,,,,...,,,,,,,,,,https://uzraudziba.bank.lv/en//market/payment-...
6,,,,,,"AS ""Industra Bank""",,,,,...,,,,,,,,,,https://uzraudziba.bank.lv/en//market/payment-...
7,,,,,,AS Magnetiq Bank,,,,,...,,,,,,,,,,https://uzraudziba.bank.lv/en//market/payment-...
8,,,,,,"AS ""SEB banka""",,,,,...,,,,,,,,,,https://uzraudziba.bank.lv/en//market/payment-...
9,,,,,,BluOr Bank AS,,,,,...,,,,,,,,,,https://uzraudziba.bank.lv/en//market/payment-...
